# Travel & Hospitality Dynamic Pricing using Reinforcement Learning

## Project Objective

This project develops an intelligent pricing agent using Reinforcement Learning (RL) to maximize total revenue from finite inventory, such as airline seats or hotel rooms.

Unlike traditional pricing strategies, the RL agent continuously learns optimal pricing decisions by interacting with a simulated booking environment.

The project follows the Markov Decision Process (MDP) framework and compares Reinforcement Learning approaches against conventional pricing baselines.

## Notebook Objectives

In this notebook we will:

- Understand the Dynamic Pricing problem
- Define the Reinforcement Learning environment
- Design the Markov Decision Process (MDP)
- Create a custom Gymnasium environment
- Validate environment behavior before training any RL agent

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [2]:
import gymnasium as gym
import numpy as np
import torch
import matplotlib.pyplot as plt

print("Gymnasium:", gym.__version__)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

Gymnasium: 1.3.0
NumPy: 2.5.1
PyTorch: 2.13.0+cpu


# Markov Decision Process (MDP)

Dynamic pricing can be formulated as a Markov Decision Process (MDP), where an agent interacts with a simulated booking environment to maximize cumulative revenue.

An MDP consists of:

- **State (S):** Current situation of the environment.
- **Action (A):** Pricing decision made by the agent.
- **Reward (R):** Revenue earned after taking an action.
- **Transition:** Movement from one state to another based on customer demand.
- **Policy (π):** Strategy used by the agent to choose prices.

## Problem Definition

The objective is to maximize the total revenue generated before the booking window closes.

The pricing agent must balance two competing objectives:

- Maintain higher prices to maximize profit per booking.
- Lower prices when necessary to avoid unsold inventory before departure.

Each episode represents one complete booking season.

## Environment Components

### State

The environment state is represented as:

(State) = (Remaining Inventory, Days Until Departure)

Example:

Inventory = 15 seats

Days Remaining = 8

State = (15, 8)

---

### Actions

The agent chooses one pricing level each day.

For simplicity, we define five pricing options:

Action 0 → ₹80

Action 1 → ₹100

Action 2 → ₹120

Action 3 → ₹140

Action 4 → ₹160

---

### Reward

If a customer purchases:

Reward = Selected Price

Otherwise:

Reward = 0

The total episode reward equals the total revenue earned throughout the booking season.

---

### Episode Ends When

- All inventory has been sold, OR
- Departure day arrives.

In [3]:
# Environment Constants

INITIAL_INVENTORY = 20
BOOKING_HORIZON = 15

PRICE_LEVELS = [80, 100, 120, 140, 160]

print("Initial Inventory:", INITIAL_INVENTORY)
print("Booking Horizon:", BOOKING_HORIZON)
print("Available Prices:", PRICE_LEVELS)

Initial Inventory: 20
Booking Horizon: 15
Available Prices: [80, 100, 120, 140, 160]


# Demand Model

Customer demand is modeled probabilistically.

Purchase probability depends on:

- Selected price
- Time remaining before departure

The model follows realistic market behavior:

- Lower prices increase demand.
- Higher prices decrease demand.
- Demand generally increases as the departure date approaches.

Randomness is introduced to simulate real customer behavior.

# Building the Custom Gymnasium Environment

In this section, we implement a custom Gymnasium environment that simulates a booking season.

The environment follows the Gymnasium API by implementing:

- `__init__()`
- `reset()`
- `step()`

The agent interacts with this environment by selecting a pricing action each day, while the environment simulates customer purchasing behavior.

In [4]:
from src.environment import DynamicPricingEnv

In [5]:
env = DynamicPricingEnv()

state, info = env.reset()

print("Initial State:", state)

Initial State: [20 15]


In [6]:
next_state, reward, terminated, truncated, info = env.step(2)

print("Next State:", next_state)
print("Reward:", reward)
print("Terminated:", terminated)

Next State: [20 14]
Reward: 0
Terminated: False


In [7]:
state, info = env.reset()

done = False

while not done:

    action = env.action_space.sample()

    state, reward, terminated, truncated, info = env.step(action)

    env.render()

    done = terminated or truncated

Inventory: 19 | Days Left: 14
Inventory: 19 | Days Left: 13
Inventory: 18 | Days Left: 12
Inventory: 17 | Days Left: 11
Inventory: 16 | Days Left: 10
Inventory: 16 | Days Left: 9
Inventory: 15 | Days Left: 8
Inventory: 14 | Days Left: 7
Inventory: 13 | Days Left: 6
Inventory: 12 | Days Left: 5
Inventory: 11 | Days Left: 4
Inventory: 10 | Days Left: 3
Inventory: 9 | Days Left: 2
Inventory: 9 | Days Left: 1
Inventory: 8 | Days Left: 0


# Stochastic Customer Demand

Real customers do not always purchase a product at the same price. Instead, purchasing behavior is probabilistic.

The demand model used in this project assumes:

- Lower prices attract more customers.
- Higher prices reduce demand.
- As the departure date approaches, urgency increases, making customers more likely to purchase.

A random number generator is used to simulate realistic customer behavior.

In [8]:
from src.demand import purchase_probability

prices = [80, 100, 120, 140, 160]

for price in prices:
    print(f"Price: {price} | Probability: {purchase_probability(price, 15):.2f}")

Price: 80 | Probability: 0.90
Price: 100 | Probability: 0.75
Price: 120 | Probability: 0.60
Price: 140 | Probability: 0.40
Price: 160 | Probability: 0.25


In [9]:
for price in prices:
    print(f"Price: {price} | Probability: {purchase_probability(price, 2):.2f}")

Price: 80 | Probability: 0.99
Price: 100 | Probability: 0.99
Price: 120 | Probability: 0.86
Price: 140 | Probability: 0.66
Price: 160 | Probability: 0.51


In [10]:
env = DynamicPricingEnv()

state, info = env.reset()

done = False

total_reward = 0

while not done:

    action = env.action_space.sample()

    state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward

    print(
        f"Price: {info['price']} | "
        f"Prob: {info['probability']:.2f} | "
        f"Purchased: {info['purchase']} | "
        f"Reward: {reward}"
    )

    env.render()

    done = terminated or truncated

print("\nTotal Revenue:", total_reward)

Price: 120 | Prob: 0.60 | Purchased: True | Reward: 120
Inventory: 19 | Days Left: 14
Price: 80 | Prob: 0.92 | Purchased: False | Reward: 0
Inventory: 19 | Days Left: 13
Price: 120 | Prob: 0.64 | Purchased: True | Reward: 120
Inventory: 18 | Days Left: 12
Price: 100 | Prob: 0.81 | Purchased: True | Reward: 100
Inventory: 17 | Days Left: 11
Price: 120 | Prob: 0.68 | Purchased: True | Reward: 120
Inventory: 16 | Days Left: 10
Price: 160 | Prob: 0.35 | Purchased: True | Reward: 160
Inventory: 15 | Days Left: 9
Price: 140 | Prob: 0.52 | Purchased: False | Reward: 0
Inventory: 15 | Days Left: 8
Price: 100 | Prob: 0.89 | Purchased: True | Reward: 100
Inventory: 14 | Days Left: 7
Price: 100 | Prob: 0.91 | Purchased: True | Reward: 100
Inventory: 13 | Days Left: 6
Price: 120 | Prob: 0.78 | Purchased: True | Reward: 120
Inventory: 12 | Days Left: 5
Price: 100 | Prob: 0.95 | Purchased: True | Reward: 100
Inventory: 11 | Days Left: 4
Price: 100 | Prob: 0.97 | Purchased: True | Reward: 100
Invento